<a href="https://colab.research.google.com/github/ChrisMouahe/TTA-Bootcamp-GenAI/blob/main/DailyChallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip /content/'Airplane Crashes and Fatalities upto 2023'.zip


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer



#Importation et nettoyage des données
df = pd.read_csv('/content/Airplane_Crashes_and_Fatalities_Since_1908_t0_2023.csv', encoding='latin1')
df.head()


In [ ]:
f"Shape of the dataset : {df.shape}"

airplane = df.copy()
airplane.drop_duplicates()

#Décompte et calcul du pourcentage des données manquantes
missing_count = airplane.isnull().sum()
missing_pct = (missing_count * 100 / len(airplane))
missing_df = pd.DataFrame({'missing_count': missing_count, 'missing_pct': missing_pct})
missing_df.sort_values(by='missing_pct', ascending=False)

cols_to_drop = missing_df[missing_df["missing_pct"] > 50].index

airplane_new = airplane.drop(columns=cols_to_drop)

#Traitement des valeurs manquantes des variables numériques
num_cols = airplane_new.select_dtypes(include=np.number).columns

for col in num_cols:
    airplane_new[col] = airplane_new[col].fillna(airplane_new[col].median())

#Traitement des valeurs des variables catégorielles
cat_cols = airplane_new.select_dtypes(include="object").columns

for col in cat_cols:
    airplane_new[col] = airplane_new[col].fillna(airplane_new[col].mode()[0])


#conversion des dates
airplane_new["Date"] = pd.to_datetime(airplane_new["Date"])
airplane_new["Year"] =airplane_new["Date"].dt.year
airplane_new["Month"] = airplane_new["Date"].dt.month
airplane_new["Decade"] = (airplane_new["Year"] // 10) * 10

airplane_new.head()


Analyse exploratoire des données

In [ ]:
#Nombre total d'accidents
total_accidents = len(airplane_new)
print(f"Nombre total d'accidents : {total_accidents}")

#Nombre total des décès
total_deaths = airplane_new["Fatalities"].sum()
print(f"Nombre total des décès : {total_deaths}")

#Taux de survie
airplane_new["survival_rate"] = (
    (airplane_new["Aboard"] - airplane_new["Fatalities"]) / airplane_new["Aboard"]
) * 100
print(f"Taux de survie : {airplane_new['survival_rate'].mean():.2f}%")


Analyse de la fréquence des incidents

In [ ]:
accidents_year = airplane_new.groupby("Year").size()

plt.figure(figsize=(12,6))

accidents_year.plot()

plt.title("Nombre d'accidents par année")
plt.xlabel("Année")
plt.ylabel("Nombre d'accidents")

plt.show()

Analyse statistique des distributions avec SciPy

In [ ]:
#Distribution des décès
  #Calcul des statistiques clées des décès
fatalities = airplane_new["Fatalities"]
fatalities.mean()
fatalities.median()
fatalities.std()

fatalities.describe()

In [ ]:
#Distribution du taux de survie
survival_rate = airplane_new["survival_rate"]

survival_rate.describe()

In [ ]:
#Test d'hypothèse : comparer le nombre moyen de décès au cours de différentes décennies
old = airplane_new[airplane_new["Decade"] < 1980]["Fatalities"]

recent = airplane_new[airplane_new["Decade"] >= 1980]["Fatalities"]

t_stat, p_value= stats.ttest_ind(old, recent, equal_var=False)

print("t =", t_stat)
print("p =", p_value)

Visualisation

In [ ]:
#Histogramme des décès
plt.figure(figsize=(10,6))

sns.histplot(
    airplane_new["Fatalities"],
    bins=25,
    kde=True
)

plt.title("Distribution des décès")

plt.show()

In [ ]:
#Accidents par région
accidents_count = (airplane_new["Location"].value_counts().head(10))

plt.figure(figsize=(10,6))

sns.barplot(
    x=accidents_count.values,
    y=accidents_count.index
)

plt.title("Top 10 des régions les plus touchées")

plt.show()

In [ ]:
#Taux de srvie par décennie
plt.figure(figsize=(10,6))

sns.boxplot(
    x=airplane_new["Decade"],
    y=airplane_new["survival_rate"]
)

plt.title("Taux de survie par décennie")

plt.show()


Analyse :

Principaux résultats
Le dataset contient 4998 accidents aériens enregistrés.
Le nombre total de décès est de 111732.
Les accidents étaient plus fréquents entre 1940 et 1950.
Une diminution progressive des accidents est observable après les années 1990.
Le taux de survie moyen est de 18.23%.
Les décès présentent une forte dispersion (écart-type élevé).
Le test statistique montre une différence significative entre les décennies étudiées.

Cocnclusion :
L'analyse met en évidence une évolution importante de la sécurité aérienne au fil du temps. Bien que certains accidents aient causé un nombre élevé de décès, la tendance globale suggère une amélioration progressive des standards de sécurité et une réduction du risque au cours des dernières décennies.

# Utilisation des bibliothèques :

*Pandas :*

* Chargement des données
* Nettoyage
* Agrégation
* Création de variables

*NumPy*

* Calculs numériques
* Manipulation des tableaux

*SciPy*

* Tests statistiques
* Analyse des distributions


*Matplotlib & Seaborn*
* Visualisation des
* tendances
* Histogrammes
* Diagrammes en barres
* Séries temporelles